In [1]:
#download and install langchain dependencies
!pip install -qU langchain-community faiss-cpu langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# importing all libraries needed
import faiss #for faiss vector database
from langchain_community.docstore.in_memory import InMemoryDocstore #for allocating vector database storage (in RAM)
from langchain_community.vectorstores import FAISS #for langchain - faiss vector database connector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI #for embedding and LLM service
from sentence_transformers import CrossEncoder #import crossencoder for reranker
from pydantic import BaseModel, Field #pydantic BaseModel and Field to create a data validation and data type definition
from typing import List #for list type

In [3]:
import getpass #for input password interface
import os #for read and manage python directory path and environment
from google.colab import userdata

#input required credentials
if not os.environ.get("OPENAI_API_KEY"):
  try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  except:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [4]:
#define embedding model and llm from openai provider
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

In [5]:
#define path for load faiss index
path_faiss = "/content/drive/MyDrive/Purwadhika/Update Materi Langchain/20. Advance RAG and Vector Databases/faiss_index"

#create vector store with saved index
vector_store = FAISS.load_local(
    path_faiss, embeddings, allow_dangerous_deserialization=True
)

In [6]:
#retrieve relevant documents from vector database
query="Men's jacket"
documents = vector_store.similarity_search(query=query, k=10)
documents

[Document(id='e0f0e5d360b14e137e5b438cd99f5849', metadata={'uniq_id': 'e0f0e5d360b14e137e5b438cd99f5849', 'product_name': "Lee Mark Men's Solid Formal Shirt"}, page_content="Lee Mark Men's Solid Formal Shirt\n                         Price: Rs. 599\n\t\t\t\t\n\t\t\tLee Mark Brand Casual Shirts Are A Fashion Essential This Season And This Uber Cool Piece Is Just The Right Pick. This Shirt Features Long Sleeves, Ponit Collar With A Full Buttoned Chest Placket For Added Ease And A Curved Hem That Completes The Look. Pair This Piece With Chinos And Loafers For A Stylish Laid-Back Look.\nLee Mark Brand Casual Shirts Are A Fashion Essential This Season And This Uber Cool Piece Is Just The Right Pick. This Shirt Features Long Sleeves, Ponit Collar With A Full Buttoned Chest Placket For Added Ease And A Curved Hem That Completes The Look. Pair This Piece With Chinos And Loafers For A Stylish Laid-Back Look."),
 Document(id='fd41178d3e845364fce52914580fd852', metadata={'uniq_id': 'fd41178d3e845

#Pre-Trained Reranker Model

In [7]:
# Load the model, here we use the model
model = CrossEncoder("mixedbread-ai/mxbai-rerank-large-v1")

config.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [8]:
docs = [d.page_content for d in documents] #get page_content for each document
results = model.rank(query, docs, return_documents=True, top_k=3) #use reranker and take top-3
results

[{'corpus_id': 7,
  'score': np.float32(0.40433145),
  'text': "Key Features of Wake Up Competition Full Sleeve Striped Men's Sweatshirt Suitable For: Western Wear Hooded Pockets: Kangaroo Pockets at Front,Wake Up Competition Full Sleeve Striped Men's Sweatshirt Price: Rs. 748 To perfect an easy going look, you must get this navy coloured hoodie from Wake Up Competition. fleece fabric, this trendy sweatshirt will be one of your smartest and worthiest picks till date. Team it over a pair of blue jeans complemented by sneakers to attain a chilled out look when you go to hang out with friends.,Specifications of Wake Up Competition Full Sleeve Striped Men's Sweatshirt Sweatshirt Details Sleeve Full Sleeve Hooded Yes Reversible No Fabric Fleece Pockets Kangaroo Pockets at Front General Details Pattern Striped Occasion Casual Ideal For Men's In the Box 1 Sweatshirt Additional Details Style Code 16-725-NAVY Fabric Care Do Not Bleach, Wash with Similar Colors, Gentle Machine Wash Cold at 30?C,

#LLM As Reranker

In [9]:
#create prompt to make llm acts and responses as a reranker
prompt = f"""
Query: {query}

Documents:
{docs}

Task: Rank the documents from most to least relevant to the query.
"""
response = llm.invoke(prompt) #generate response from llm
print(response.content) #show the result

Based on the query "Men's jacket," the ranked relevance of the documents is as follows:

1. **Specifications of Mizuno Lightweight Hoody Solid Men's Wind Cheater** - This document specifically mentions a "wind cheater," which can be considered a type of jacket.
  
2. **Key Features of Roadster Men's Zipper Solid Cardigan** - While it's a cardigan rather than a jacket, it is an outerwear piece and thus relevant.

3. **Wake Up Competition Full Sleeve Striped Men's Sweatshirt** - Though this is a sweatshirt, it is also an outerwear item that might be considered in discussions surrounding jackets.

4. **All other documents** - The remaining documents primarily focus on shirts, jeans, and other categories of clothing that are not closely related to jackets, thus they are ranked equally low in relevance and can be placed at the bottom.

So the final ranking is as follows:
1. Specifications of Mizuno Lightweight Hoody Solid Men's Wind Cheater
2. Key Features of Roadster Men's Zipper Solid Car

In [10]:
#make llm as reranker responses with structured output
class DocumentAttribute(BaseModel): #detail information for responses
  doc_number: str = Field(..., description="Original document number")
  score: float = Field(..., description="relevant score, interval 0 to 1")

class ResponseOutput(BaseModel): #output template
  reranker_result: List[DocumentAttribute] = Field(..., description="reranker result") #template from DocumentAttribute class for detail of each document


llm_with_structured__output = llm.with_structured_output(ResponseOutput) #method to give information and instruction to llm to follow the output template
response = llm_with_structured__output.invoke(prompt) #generate response
response.model_dump() #transform the output from pydantic object into json/dictionary


{'reranker_result': [{'doc_number': '3', 'score': 0.8},
  {'doc_number': '4', 'score': 0.75},
  {'doc_number': '8', 'score': 0.6},
  {'doc_number': '1', 'score': 0.4},
  {'doc_number': '2', 'score': 0.4},
  {'doc_number': '5', 'score': 0.3},
  {'doc_number': '6', 'score': 0.2},
  {'doc_number': '7', 'score': 0.2},
  {'doc_number': '9', 'score': 0.1}]}